<a href="https://colab.research.google.com/github/jenipherorishaba2-ship-it/nlp-project-group-Jenipher1-and-Racheal2/blob/main/NLP_End_of_Module_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-of-Module Project: From NLP Model to Live Apps

**Work in your group of 2. Deadline: 30th October 2026.**

You already know the NLP basics from the tutorial. This notebook walks you through everything you need to ship **two live apps**:

| Part | What you build |
|---|---|
| **Part A** | Your sentiment model on the Hugging Face Hub, a Gradio demo live on Spaces, and a FastAPI `/predict` API |
| **Part B** | An LLM-powered chatbot that answers questions from your group's PDF, live on Spaces |

Run the cells in order. Wherever you see `# TODO`, change the value for your group.

**Before you start:**
- A Hugging Face account and an access token with **write** permission (Settings → Access Tokens)
- Add the token to Colab: click the 🔑 **Secrets** icon on the left, name it `HF_TOKEN`, paste the token, and switch on notebook access
- One GitHub repo for your group, with both members as collaborators

## 0 · Setup

In [1]:
!pip install -q transformers datasets huggingface_hub gradio fastapi uvicorn pypdf sentence-transformers

In [1]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")      # read from Colab Secrets, never paste tokens in code
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: Racheal26


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


---
# PART A · Ship your sentiment model

## A1 · Load your fine-tuned model

Use a sentiment analysis model. As a group, compare your two models and **pick the one with the higher test F1**.

If your model is saved in a folder (e.g. `/content/my_model`), point `LOCAL_MODEL_DIR` to it. If you need to re-train, re-run your notebook and save with `trainer.save_model("/content/my_model")` and `tokenizer.save_pretrained("/content/my_model")`.

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

LOCAL_MODEL_DIR = "/content/drive/MyDrive/my_model"   # TODO: path to your saved model


tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(clf("Terrible experience, I want my money back."))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.8636758327484131}]


## A2 · Push your model to the Hugging Face Hub

This makes your model public so any app (or anyone) can load it with one line.

In [5]:
MODEL_NAME = "group4-sentiment-model"
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

model.push_to_hub(MODEL_ID)
tokenizer.push_to_hub(MODEL_ID)
print("Model page:", f"https://huggingface.co/{MODEL_ID}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...c2zroj3/model.safetensors:   0%|          |  575kB /  268MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Model page: https://huggingface.co/Jenicoder/group4-sentiment-model


### Write a model card

A model card tells people what your model does and where it fails. Fill in your real numbers.

In [6]:
from huggingface_hub import ModelCard

card_text = f"""---
language: en
pipeline_tag: text-classification
tags: [sentiment-analysis]
---

# {MODEL_NAME}

Sentiment classifier (positive / negative) fine-tuned from `distilbert-base-uncased`.

**Group members:** Orishaba Jenipher and Nalweyiso Racheal
**Dataset:** stanfordnlp/imdb (movie reviews)

## Results on the test set
| Metric | Score |
|---|---|
| Accuracy | 0.90 |
| F1 | 0.90 |

## Example predictions
- "Terrible experience, I want my money back." → NEGATIVE (0.86)
- "The food was amazing!" → POSITIVE (0.99)
- "It was okay, nothing special." → NEGATIVE (0.73)

## Limitations
Trained on English movie reviews only; may struggle with sarcasm, Luganda text, and mixed-language reviews.
"""
ModelCard(card_text).push_to_hub(MODEL_ID)
print("Model card uploaded.")


Model card uploaded.


### Check: load it back from the Hub (this is how your apps will load it)

In [7]:
hub_clf = pipeline("text-classification", model=MODEL_ID)
hub_clf("I really enjoyed this class!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9817610383033752}]

## A3 · Build a Gradio demo

Gradio turns a Python function into a web app. First test it here in Colab.

In [8]:
import gradio as gr

def predict(text):
    if not text.strip():
        return {}
    result = hub_clf(text)[0]
    score = result["score"]
    # Gradio's Label component shows a confidence bar for each class
    other = "NEGATIVE" if result["label"] == "POSITIVE" else "POSITIVE"
    return {result["label"]: score, other: 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    description="Type a sentence and see if it is positive or negative.",
    examples=["The food was amazing!", "Worst service ever.", "It was okay, nothing special."],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6df879a7fc67a6cbdf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> If your labels show as `LABEL_0` / `LABEL_1`, your model has no label names. Map them in `predict()`, e.g. `{'LABEL_0': 'NEGATIVE', 'LABEL_1': 'POSITIVE'}`.

## A4 · Deploy the demo to Hugging Face Spaces

A **Space** hosts your app for free with a public link. The cell below writes the two files a Space needs and uploads them.

In [9]:
%%writefile app.py
import gradio as gr
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id
clf = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {}
    r = clf(text)[0]
    other = "NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"
    return {r["label"]: r["score"], other: 1 - r["score"]}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    examples=["The food was amazing!", "Worst service ever."],
)
demo.launch()

Writing app.py


In [10]:
%%writefile requirements.txt
transformers
torch

Writing requirements.txt


In [13]:
from huggingface_hub import HfApi
api = HfApi()

# Replace the placeholder model id inside app.py with yours
app_code = open("app.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID)
open("app.py", "w").write(app_code)

SPACE_ID = f"{HF_USERNAME}/group1-sentiment-demo"   # TODO: change group number
api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=SPACE_ID, repo_type="space")
print("Your Space (takes 2-5 minutes to build):", f"https://huggingface.co/spaces/{SPACE_ID}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6adf6246-4c9ca5150f837d962db1a2f6;f30c3e3b-5f74-4165-8b98-04a81e5acae7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

✅ **Checkpoint:** open the Space link. When it says *Running*, test it, then send the link to another group to test.

## A5 · Turn your model into an API with FastAPI

A demo is for **people**. An API is for **other programs** (a mobile app, a website, a WhatsApp bot). They send JSON, your API sends JSON back.

We'll run it here in Colab so you can test it. On your laptop you run the same file with `uvicorn main:app --reload` and open `http://127.0.0.1:8000/docs`.

In [3]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id

app = FastAPI(title="Sentiment API")

# Load the model ONCE when the app starts, not on every request (that would be very slow)
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))

Writing main.py


In [4]:
# Put your model id into main.py, then start the API in the background
open("main.py", "w").write(open("main.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID))

import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)   # give it time to load the model
print("API started")

NameError: name 'MODEL_ID' is not defined

In [18]:
# Write main.py with your exact model ID
with open("main.py", "w") as f:
    f.write('''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "Jenicoder/group4-sentiment-model"

app = FastAPI(title="Sentiment API")
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))
''')

import subprocess, time
try:
    server.terminate()  # stop old server if running
except:
    pass

server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(35)   # wait 35 seconds
print("API started")


API started


In [19]:
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

for sentence in ["I love studying data science!", "This assignment is so hard.", ""]:
    r = requests.post(f"{BASE}/predict", json={"text": sentence})
    print(r.status_code, r.json())


{'status': 'ok'}
200 {'label': 'POSITIVE', 'score': 0.9687}
200 {'label': 'POSITIVE', 'score': 0.5516}
400 {'detail': 'Text cannot be empty'}


In [20]:
server.terminate()   # stop the API when you are done

✅ **Checkpoint:** you should see `200` with a label and score for the two sentences, and `400` for the empty one.

**Push to GitHub:** `main.py`, `requirements.txt` (add `fastapi` and `uvicorn`) and a README with:
- how to install: `pip install -r requirements.txt`
- how to run: `uvicorn main:app --reload`
- an example request and response

**Stretch (optional):** add a `Dockerfile` and deploy the API as a Docker Space.

---
# PART B · Build an LLM chatbot over your group's PDF

## B1 · Talk to an LLM from Python

We use **Hugging Face Inference Providers** so you can use the same `HF_TOKEN` (it includes a small amount of free usage each month). The code style is the same as OpenAI's, so it transfers to other providers.

> If the model below is unavailable, open [huggingface.co/models](https://huggingface.co/models?inference_provider=all&pipeline_tag=text-generation), filter by *Inference Providers*, and pick another chat model.

In [6]:
import torch
from transformers import pipeline

print("Loading LLM on your T4 GPU (takes ~15 seconds)...")
llm_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device="cuda"
)

def ask_llm(messages, max_tokens=400):
    output = llm_pipe(messages, max_new_tokens=max_tokens, do_sample=False)
    # Extract the assistant's reply text
    return output[0]["generated_text"][-1]["content"].strip()

# Test it
answer = ask_llm([
    {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
    {"role": "user", "content": "What is natural language processing?"},
])

print("\nAI Answer:\n", answer)


Loading LLM on your T4 GPU (takes ~15 seconds)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


AI Answer:
 Natural Language Processing (NLP) is the branch of artificial intelligence that enables computers to understand, interpret, and generate human language.


### Compare: your fine-tuned model vs the LLM

The LLM can do sentiment analysis with **no training at all**, just a prompt. But is it better, faster, or cheaper?

In [7]:
import time, json
from transformers import pipeline

# Make sure your fine-tuned model is loaded
try:
    hub_clf
except NameError:
    hub_clf = pipeline("text-classification", model="Jenicoder/group4-sentiment-model")

test_sentences = [
    "The delivery was quick and the packaging was great.",
    "I waited two hours and nobody helped me.",
    "Not bad, but I expected more for the price.",
    "Absolutely brilliant lecture today!",
    "The app keeps crashing every time I open it.",
]

for s in test_sentences:
    t0 = time.time(); mine = hub_clf(s)[0]["label"]; t_mine = time.time() - t0
    t0 = time.time()
    llm_label = ask_llm([
        {"role": "system", "content": "Classify the sentiment. Reply with ONE word: POSITIVE or NEGATIVE."},
        {"role": "user", "content": s},
    ], max_tokens=10).strip()
    t_llm = time.time() - t0
    print(f"{s[:45]:<47} mine={mine:<9} ({t_mine:.2f}s)  llm={llm_label:<9} ({t_llm:.2f}s)")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

The delivery was quick and the packaging was    mine=POSITIVE  (0.12s)  llm=POSITIVE  (0.14s)
I waited two hours and nobody helped me.        mine=NEGATIVE  (0.02s)  llm=NEGATIVE  (0.16s)
Not bad, but I expected more for the price.     mine=NEGATIVE  (0.01s)  llm=NEGATIVE  (0.13s)
Absolutely brilliant lecture today!             mine=POSITIVE  (0.01s)  llm=POSITIVE  (0.13s)
The app keeps crashing every time I open it.    mine=NEGATIVE  (0.01s)  llm=NEGATIVE  (0.14s)


### Getting structured JSON from an LLM

In [8]:
complaint = "I paid for MTN data bundles yesterday but they never came. This is the third time! Fix it today."

reply = ask_llm([
    {"role": "system", "content": (
        "Extract information from customer complaints. "
        "Reply ONLY with JSON with keys: product, issue, urgency (low, medium or high). No other text."
    )},
    {"role": "user", "content": complaint},
])
print(reply)
data = json.loads(reply.strip().strip("`").removeprefix("json"))
data

{
  "product": "MTN data bundles",
  "issue": "Delivery of purchased products",
  "urgency": "high"
}


{'product': 'MTN data bundles',
 'issue': 'Delivery of purchased products',
 'urgency': 'high'}

💬 **Discuss in your group:** When would you use your small fine-tuned model instead of an LLM? Think about speed, cost, privacy and internet access.

## B2 · RAG: answer questions from your PDF

LLMs don't know about *your* documents and may invent answers. **Retrieval-Augmented Generation (RAG)** fixes this:

1. Split the document into small chunks
2. Turn each chunk into an embedding (like the BERT embeddings in the tutorial)
3. When a question comes in, find the most similar chunks
4. Give those chunks to the LLM and tell it to answer **only** from them

### Upload your group's PDF
Each group uses a **different** document (course handbook, hostel rules, a health guide, a mobile-money FAQ, a farming guide, ...). Click the 📁 folder icon on the left and upload it, or run the cell below.

In [10]:
!pip install -q fpdf2
from fpdf import FPDF

# Create a clean 5-page University Handbook PDF
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

pages_data = [
    ("Section 1: General Hostel Rules and Visitor Regulations",
     "All registered residents must observe the hostel code of conduct at all times. Visiting hours for external guests are strictly between 10:00 AM and 8:00 PM on weekdays, and between 10:00 AM and 9:00 PM on weekends and public holidays. All non-resident visitors must register at the main security desk and deposit a valid national identity card. Overnight visitors are strictly prohibited in all university residence halls without prior written permission from the Dean of Students. Quiet hours begin promptly at 10:00 PM every night to maintain a peaceful study environment."),

    ("Section 2: Room Maintenance and Prohibited Electrical Items",
     "Bi-weekly room cleanliness inspections are conducted every Wednesday morning between 9:00 AM and 11:00 AM by the hostel management. High-power appliances including immersion water heaters, hotplates, electric charcoal stoves, and unauthorized room heaters are strictly prohibited due to electrical safety and fire hazard prevention. Permitted personal items include laptops, mobile phones, study lamps, and low-power electric kettles. Any unauthorized electrical appliances found during room inspections will be confiscated immediately."),

    ("Section 3: Health Services and Emergency Evacuation Procedures",
     "The University Health Centre operates Monday through Friday from 8:00 AM to 5:00 PM for all registered students. In the event of a medical emergency outside clinic hours, students should contact the on-duty emergency medical service at 0800-111-999. In case of a fire or emergency alarm, all occupants must evacuate the hostel building immediately through the emergency stairs and assemble at Fire Assembly Point B located at the university sports grounds. First aid supplies are kept in the floor warden office."),

    ("Section 4: Examination Regulations and Missed Exam Procedures",
     "Regarding examination attendance, what happens if a student misses an exam: Any student who misses a scheduled examination due to severe illness or hospitalization must submit an official medical certificate signed by a recognized medical officer to the Academic Registrar within 48 hours of the missed exam. Unexcused absence from an examination results in an automatic retake grade of zero. Students must arrive at examination halls at least 20 minutes before the scheduled start time. Mobile phones and electronic notes are strictly forbidden inside examination venues."),

    ("Section 5: Tuition Deadlines and Caution Deposit Refund Policy",
     "All students must clear at least 50 percent of their semester tuition by the end of Week 4, and 100 percent full clearance must be completed by Week 8 to receive official examination permits. Late payments made after Week 8 incur an administrative penalty charge of 10 percent of the outstanding balance. The hostel caution deposit of 200,000 UGX is refundable at the end of the academic year, following a formal room clearance inspection conducted by the Hostel Warden ensuring zero property damage.")
]

for title, body in pages_data:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, title, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 12)
    pdf.multi_cell(0, 8, body)

PDF_PATH = "hostel_guide.pdf"
pdf.output(PDF_PATH)
print("Using generated document:", PDF_PATH)


Using generated document: hostel_guide.pdf


In [12]:
!pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
print(f"{len(pages)} pages. First 300 characters of page 1:\n")
print(pages[0][1][:300])


5 pages. First 300 characters of page 1:

Section 1: General Hostel Rules and Visitor Regulations
All registered residents must observe the hostel code of conduct at all times. Visiting hours for
external guests are strictly between 10:00 AM and 8:00 PM on weekdays, and between 10:00 AM
and 9:00 PM on weekends and public holidays. All non-r


### Step 1 · Split into chunks (keeping the page number)

In [14]:
def chunk_pages(pages, chunk_size=500, overlap=100):
    chunks = []
    for page_num, text in pages:
        text = " ".join(text.split())          # clean up extra spaces and line breaks
        start = 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if len(piece) > 50:                # skip tiny leftovers
                chunks.append({"page": page_num, "text": piece})
            start += chunk_size - overlap      # overlap so sentences aren't cut in half
    return chunks

chunks = chunk_pages(pages)
print(len(chunks), "chunks")
print(chunks[0])


10 chunks
{'page': 1, 'text': 'Section 1: General Hostel Rules and Visitor Regulations All registered residents must observe the hostel code of conduct at all times. Visiting hours for external guests are strictly between 10:00 AM and 8:00 PM on weekdays, and between 10:00 AM and 9:00 PM on weekends and public holidays. All non-resident visitors must register at the main security desk and deposit a valid national identity card. Overnight visitors are strictly prohibited in all university residence halls without prior written '}


### Step 2 · Embed the chunks

`sentence-transformers` models are trained so that sentences with similar meaning get similar vectors, which makes them better for search than the mean-pooled BERT from the tutorial.

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(chunk_vectors.shape)   # (number of chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(10, 384)


### Step 3 · Retrieve the most relevant chunks for a question

In [16]:
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec            # cosine similarity (vectors are normalised)
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]

question = "What happens if a student misses an exam?"   # TODO: a question about YOUR document
for hit in retrieve(question):
    print(f"page {hit['page']}  score={hit['score']:.2f}  {hit['text'][:120]}...")

page 4  score=0.77  Section 4: Examination Regulations and Missed Exam Procedures Regarding examination attendance, what happens if a studen...
page 4  score=0.55  mination results in an automatic retake grade of zero. Students must arrive at examination halls at least 20 minutes bef...
page 5  score=0.39  Section 5: Tuition Deadlines and Caution Deposit Refund Policy All students must clear at least 50 percent of their seme...


### Step 4 · Let the LLM answer from those chunks only

In [17]:
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. "
    "Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document."
)

def rag_answer(question, k=3):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)
    return ask_llm([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

print(rag_answer(question))

If a student misses an exam due to severe illness or hospitalization, they must submit an official medical certificate signed by a recognized medical officer to the Academic Registrar within 48 hours of the missed exam. This allows for consideration of excused absences. However, unexcused absence from an examination results in an automatic retake grade of zero. Students must also arrive at examination halls at least 20 minutes before the scheduled start time.


### Test it properly

Write **5 questions**: 3 that your document answers and 2 that it does **not** answer. A good RAG system says "I don't know" for the last 2 instead of inventing an answer.

In [ ]:
test_questions = [
    "TODO question 1 (answer is in the document)",
    "TODO question 2 (answer is in the document)",
    "TODO question 3 (answer is in the document)",
    "Who won the 2010 World Cup?",                  # NOT in the document
    "TODO question 5 (NOT in the document)",
]
for q in test_questions:
    print("Q:", q)
    print("A:", rag_answer(q), "\n")

📝 Record in your README: for each question, was the right page retrieved? Was the answer correct? Did it invent anything?

## B3 · Turn it into a chatbot

`gr.ChatInterface` gives you a chat window. We pass the **conversation history** to the LLM so it remembers earlier questions (e.g. *"What about the second one?"*).

In [ ]:
def chat(message, history):
    hits = retrieve(message)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)

    messages = [{"role": "system", "content": SYSTEM_PROMPT + " Be polite. Refuse questions unrelated to the document."}]
    messages += history[-6:]                    # last few turns only, to keep it short and cheap
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})
    return ask_llm(messages)

bot = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Ask our document",                   # TODO: e.g. "Hostel Rules Assistant"
    examples=["What is this document about?"],
)
bot.launch()

🧪 **Try to break it** (5 attempts): ask off-topic questions, be rude, ask it to ignore its instructions, ask something tricky. Improve `SYSTEM_PROMPT` until it behaves. Keep a short log of what broke and how you fixed it.

## B4 · Deploy the chatbot to Spaces

The Space needs your PDF, the app code, and your token stored as a **secret** (never inside the code).

In [ ]:
%%writefile chatbot_app.py
import os
import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

PDF_PATH = "document.pdf"
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document. "
    "Be polite. Refuse questions unrelated to the document."
)

llm = InferenceClient(api_key=os.environ["HF_TOKEN"])   # read from the Space secret
embedder = SentenceTransformer("all-MiniLM-L6-v2")

pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(PDF_PATH).pages)]
chunks = []
for page_num, text in pages:
    text = " ".join(text.split())
    for start in range(0, len(text), 400):
        piece = text[start:start + 500]
        if len(piece) > 50:
            chunks.append({"page": page_num, "text": piece})
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def retrieve(q, k=3):
    scores = vectors @ embedder.encode([q], normalize_embeddings=True)[0]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]

def chat(message, history):
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in retrieve(message))
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history[-6:]
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})
    r = llm.chat.completions.create(model=LLM_MODEL, messages=messages, max_tokens=400, temperature=0.2)
    return r.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages", title="Ask our document").launch()

In [ ]:
%%writefile chatbot_requirements.txt
pypdf
sentence-transformers
huggingface_hub
numpy

In [ ]:
CHAT_SPACE_ID = f"{HF_USERNAME}/group1-doc-chatbot"   # TODO: change group number

api.create_repo(CHAT_SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
api.upload_file(path_or_fileobj="chatbot_app.py", path_in_repo="app.py", repo_id=CHAT_SPACE_ID, repo_type="space")
api.upload_file(path_or_fileobj="chatbot_requirements.txt", path_in_repo="requirements.txt", repo_id=CHAT_SPACE_ID, repo_type="space")
api.upload_file(path_or_fileobj=PDF_PATH, path_in_repo="document.pdf", repo_id=CHAT_SPACE_ID, repo_type="space")

# Store the token as a SECRET in the Space (safe: not visible in the code)
api.add_space_secret(CHAT_SPACE_ID, "HF_TOKEN", HF_TOKEN)

print("Your chatbot (takes a few minutes to build):", f"https://huggingface.co/spaces/{CHAT_SPACE_ID}")

> ⚠️ Only use a document that is fine to make public. A public Space lets anyone download its files.

✅ **Checkpoint:** open the link, ask 3 questions, and send it to another group to test.

---
# 📤 Submission checklist (one submission per group on Google Classroom)

- [ ] **Model link** on the Hugging Face Hub, with a completed model card
- [ ] **Sentiment demo** Space link (works for someone outside your group)
- [ ] **GitHub repo** with the FastAPI code and a README (install, run, example request, test results, limitations, **who did what**)
- [ ] **Chatbot** Space link
- [ ] Your **RAG test results** (5 questions) and your **"what broke and how we fixed it"** log
- [ ] **No tokens or API keys** anywhere in your code or GitHub history
- [ ] Both members have commits in the repo
- [ ] Be ready for a **5-minute demo** where each member presents one part

## 🚑 Common problems

| Problem | Fix |
|---|---|
| `HF_TOKEN` not found | Add it in Colab Secrets (🔑) and switch on notebook access |
| 403 / permission error when pushing | Your token needs **write** permission |
| Space shows *Build error* | Open the **Logs** tab; usually a missing package in `requirements.txt` |
| Labels are `LABEL_0` / `LABEL_1` | Map them to names in your `predict()` function |
| LLM call fails / model not available | Pick another chat model with Inference Providers enabled |
| Chatbot invents answers | Make the system prompt stricter; check that `retrieve()` returns the right page |
| PDF text is empty | Your PDF is scanned images. Use a PDF with selectable text |